# 06 — Threshold Optimization & Decision Framework

Raw model probabilities aren't lending decisions. This notebook sweeps classification thresholds, picks an
operating point, and converts probabilities into Approve / Manual Review / Reject decisions.

Builds on the `risk_scores.csv` produced by notebook 05 (XGBoost probabilities on the holdout set) — all
threshold/decision logic itself lives in `credit_risk.models.decision`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

from credit_risk.config import settings
from credit_risk.models import assign_decision, build_scorecard, sweep_thresholds

In [ ]:
risk_scores = pd.read_csv(settings.outputs_dir / "predictions" / "risk_scores.csv")
risk_scores.head()

## Probability distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(risk_scores["probability_default"], bins=50)
plt.title("Predicted Default Probability Distribution")
plt.xlabel("Probability of Default")
plt.show()

## Threshold sweep

`sweep_thresholds` evaluates precision/recall/F1 across `np.arange(0.1, 1.0, 0.05)`.

In [ ]:
threshold_df = sweep_thresholds(risk_scores["actual"], risk_scores["probability_default"])
threshold_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.plot(threshold_df["threshold"], threshold_df["f1_score"], label="F1 Score")
plt.xlabel("Threshold")
plt.ylabel("Metric Score")
plt.title("Threshold Optimization")
plt.legend()
plt.show()

Lower thresholds prioritize catching defaults (recall) at the cost of precision; higher thresholds do the
opposite. **0.40** was selected as the balance point for this operating model.

In [ ]:
selected_threshold = 0.40
final_preds = np.where(risk_scores["probability_default"] >= selected_threshold, 1, 0)
print(classification_report(risk_scores["actual"], final_preds))

In [ ]:
cm = confusion_matrix(risk_scores["actual"], final_preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Optimized Threshold Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Decision framework

`assign_decision` maps probability to an operational decision: **< 20% → Approve**, **20-50% → Manual
Review**, **>= 50% → Reject**.

In [ ]:
risk_scores["decision"] = assign_decision(risk_scores["probability_default"])
risk_scores["decision"].value_counts()

In [ ]:
decision_default_rate = risk_scores.groupby("decision")["actual"].mean() * 100
decision_default_rate

The Reject segment's realized default rate is roughly 6x the Approve segment's — the framework is
separating risk, not just rejecting arbitrarily.

In [ ]:
portfolio_summary = risk_scores.groupby("decision").agg(
    probability_default_mean=("probability_default", "mean"),
    count=("probability_default", "count"),
    actual_mean=("actual", "mean"),
)
portfolio_summary

## Borrower scorecard

In [ ]:
scorecard = build_scorecard(risk_scores["actual"], risk_scores["probability_default"])
scorecard.head(20)

In [ ]:
predictions_dir = settings.outputs_dir / "predictions"
threshold_df.to_csv(predictions_dir / "threshold_metrics.csv", index=False)
portfolio_summary.to_csv(predictions_dir / "portfolio_summary.csv")
decision_default_rate.to_csv(predictions_dir / "decision_default_rate.csv")
scorecard.to_csv(predictions_dir / "borrower_scorecard.csv", index=False)

## Business interpretation

Converting probabilities into a three-tier decision gives underwriting a workable operational structure:
auto-approve the low-risk majority, auto-reject the clearly high-risk minority, and route the ambiguous
middle to manual review. The realized default rate by decision tier (Approve ≈ 11%, Manual Review ≈ 31%,
Reject ≈ 61% in the original run) shows the framework tracks realized outcomes, not just predicted ones —
the scorecard and portfolio summary above are the artifacts that would feed a real underwriting workflow.